# 03 — Feature Engineering & Preprocessing

**What we engineer (and why):**

| Feature | From | Rationale |
|---|---|---|
| `Hour` | `Time` | Hour-of-day (0–23). Fraud rate spikes at night. Raw `Time` (seconds since collection start) is an artifact of the 2-day window and is dropped. |
| `Amount_log` | `Amount` | `log1p` tames extreme right skew so the linear model isn't dominated by outliers. Raw `Amount` is kept in the frame for business-cost analysis but excluded from model features. |

**What we leave alone:** `V1–V28` are already PCA outputs — centered, orthogonal, informative.
Re-transforming them adds nothing.

**Scaling.** `StandardScaler` lives *inside* the model pipeline, so it is fit only on training folds
— never on validation/test (leakage-safe by construction). Trees don't need scaling but it is
harmless, and one uniform pipeline keeps training and serving identical.

**Leakage checklist applied here:**
1. Duplicates dropped **before** splitting (no identical row in train and test).
2. Feature engineering is row-wise only (no statistics learned from the full dataset).
3. Scaler and resamplers are pipeline steps → fit on training data only.
4. Threshold tuned on **validation**, test set touched exactly once.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display
from src.utils import load_config, resolve_path

config = load_config()
pd.set_option("display.max_columns", 40)

In [2]:
from src.data_loader import load_raw_data
from src.preprocessing import drop_duplicates, make_splits
from src.feature_engineering import engineer_features, get_feature_columns

df = drop_duplicates(load_raw_data(config))
df_fe = engineer_features(df, config)
feature_cols = get_feature_columns(df_fe, config)
print(f"{len(feature_cols)} model features:")
print(feature_cols)

2026-07-31 18:15:07 | INFO    | src.data_loader | Loaded raw data: 284807 rows x 31 columns


2026-07-31 18:15:07 | INFO    | src.preprocessing | Dropped 1081 duplicate rows (284807 -> 283726)


2026-07-31 18:15:07 | INFO    | src.feature_engineering | Feature engineering done: 32 columns (['hour', 'amount_log'] added, ['Time'] dropped)


30 model features:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Hour', 'Amount_log']


Sanity check — fraud rate by engineered `Hour` (the night-time spike the feature captures):

In [3]:
hourly = df_fe.groupby("Hour")["Class"].agg(n="size", frauds="sum", rate="mean")
hourly["rate_pct"] = (100 * hourly.pop("rate")).round(3)
hourly.T

Hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
n,7647.000,4208.000,3308.000,3487.000,2204.000,2988.000,4082.00,7233.000,10232.000,15767.000,16548.000,16781.000,15378.000,15323.000,16520.000,16374.000,16396.000,16130.000,16959.000,15566.000,16705.000,17629.000,15378.000,10883.000
frauds,6.000,10.000,48.000,17.000,23.000,11.000,9.00,23.000,9.000,16.000,8.000,53.000,17.000,17.000,23.000,26.000,22.000,28.000,28.000,19.000,18.000,16.000,9.000,17.000
rate_pct,0.078,0.238,1.451,0.488,1.044,0.368,0.22,0.318,0.088,0.101,0.048,0.316,0.111,0.111,0.139,0.159,0.134,0.174,0.165,0.122,0.108,0.091,0.059,0.156


## Stratified three-way split

- **Test (20%)** — untouched until final evaluation; simulates future unseen transactions.
- **Validation (16%)** — model selection + threshold tuning.
- **Train (64%)** — model fitting.

`stratify=y` keeps the 0.17% fraud rate identical in all three — with 492 frauds total, an
unstratified split could easily produce a test set with too few frauds to evaluate on.

In [4]:
splits = make_splits(df_fe, feature_cols, config)
pd.DataFrame({
    "rows": [len(splits.y_train), len(splits.y_val), len(splits.y_test)],
    "frauds": [int(s.sum()) for s in (splits.y_train, splits.y_val, splits.y_test)],
    "fraud_%": [round(100 * s.mean(), 4) for s in (splits.y_train, splits.y_val, splits.y_test)],
}, index=["train", "val", "test"])

2026-07-31 18:15:07 | INFO    | src.preprocessing | Split train: 181584 rows, 302 frauds (0.1663%)


2026-07-31 18:15:07 | INFO    | src.preprocessing | Split val  :  45396 rows,  76 frauds (0.1674%)


2026-07-31 18:15:07 | INFO    | src.preprocessing | Split test :  56746 rows,  95 frauds (0.1674%)


,rows,frauds,fraud_%
train,181584,302,0.1663
val,45396,76,0.1674
test,56746,95,0.1674
